# Heteroskedastic Ordered Probit Models with an Artificial Neural Network

**ArXivist-generated reproduction notebook**  
Paper: Jeong, J. (2024). *Heteroskedastic Ordered Probit Models with an ANN.*  
Published in: *Computational Economics*, Springer.

---

### How to use this notebook

**If running in Google Colab (recommended):**
1. Run **Cell 1** (environment check)
2. Run **Cell 2** (upload your `paper-repos.zip`) — a file picker will appear
3. Run all remaining cells in order

**If running locally (Jupyter):**
1. Run **Cell 1** (environment check)
2. Skip Cell 2 — run **Cell 2b** instead (sets local path)
3. Run all remaining cells in order

**Estimated runtime:** ~3 min on T4 GPU / ~10 min on CPU

In [1]:
# ── Cell 1: Environment Check ─────────────────────────────────────────────────
import sys, os
import torch

print(f"Python:         {sys.version.split()[0]}")
print(f"PyTorch:        {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:            {torch.cuda.get_device_name(0)}")
    print(f"VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("Running on CPU — training will be slower than paper (T4 GPU)")

IN_COLAB = 'google.colab' in sys.modules or os.path.exists('/content')
print(f"\nEnvironment:    {'Google Colab' if IN_COLAB else 'Local Jupyter'}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device:   {device}")

Python:         3.12.13
PyTorch:        2.11.0+cu128
CUDA available: True
GPU:            Tesla T4
VRAM:           15.6 GB

Environment:    Google Colab
Using device:   cuda


In [2]:
# ── Cell 2: Setup (Google Colab) ──────────────────────────────────────────────
# Upload paper-repos.zip when the file picker appears.
# To create the zip: right-click the paper-repos folder on Windows → Send to → Compressed (zip)
#
# SKIP THIS CELL if running locally — run Cell 2b instead.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, zipfile
from pathlib import Path

try:
    from google.colab import files
    print("Pick your paper-repos.zip file...")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
    print(f"\nExtracting {zip_name}...")
    with zipfile.ZipFile(zip_name, 'r') as z:
        z.extractall("/content/")

    # Find the extracted root (handles both paper-repos/ and paper-repos-main/ etc.)
    candidates = [p for p in Path("/content").iterdir()
                  if p.is_dir() and (p / "setup.py").exists()]
    if not candidates:
        raise FileNotFoundError(
            "setup.py not found after extraction. "
            "Make sure you zipped the paper-repos/ folder itself, not its contents."
        )
    PROJECT_ROOT = candidates[0]
    print(f"Project root:  {PROJECT_ROOT}")

except ImportError:
    # Not in Colab — silently skip, use Cell 2b
    print("Not in Colab. Please run Cell 2b below to set local path.")
    PROJECT_ROOT = None

if PROJECT_ROOT:
    # Add src/ to path
    src_path = str(PROJECT_ROOT / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    # Install only the packages Colab doesn't have pre-installed
    print("\nInstalling statsmodels and xgboost...")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "statsmodels>=0.14.0", "xgboost>=2.0.0"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print("Done. Ready to run!")
    else:
        print("Warning:", result.stderr[:300])

Pick your paper-repos.zip file...


Saving paper-repos.zip to paper-repos.zip

Extracting paper-repos.zip...
Project root:  /content/paper-repos

Installing statsmodels and xgboost...
Done. Ready to run!


In [ ]:
# ── Cell 2b: Setup (Local Jupyter) ───────────────────────────────────────────
# SKIP THIS CELL if running in Colab — use Cell 2 instead.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys
from pathlib import Path

# Walk upward from CWD until setup.py is found
_start = Path(os.path.abspath("")).resolve()
PROJECT_ROOT = _start
for _ in range(5):
    if (PROJECT_ROOT / "setup.py").exists():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "setup.py").exists():
    print("ERROR: Could not find setup.py.")
    print(f"  Started from: {_start}")
    print("  Launch Jupyter from inside the paper-repos/ directory.")
else:
    src_path = str(PROJECT_ROOT / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(PROJECT_ROOT)],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"hopann installed from: {PROJECT_ROOT}")
    else:
        print("Install error:", result.stderr[:300])

## Paper Overview

### The Problem
Star ratings (1 to 5) are **ordinal** — the order matters but the gap between classes is not equal.
Standard ANN classifiers with softmax output ignore this structure entirely.

### Key Innovation: OPANN
Instead of a softmax, the ANN feeds a scalar **latent index** $f(\mathbf{x}_i, \theta)$ into
an **ordered probit** log-likelihood:

$$P(y_i = j \mid \mathbf{x}_i) = \Phi(c_j - f(\mathbf{x}_i, \theta)) - \Phi(c_{j-1} - f(\mathbf{x}_i, \theta))$$

where $\Phi$ is the standard normal CDF and $c_1 < c_2 < \ldots < c_{J-1}$ are learned cutting points.

### Extension: HOPANN (Heteroskedastic)
HOPANN adds a **per-observation variance** $\sigma_i > 0$, learned by a second ANN:

$$\sigma_i = \exp(\text{VarianceANN}(\mathbf{z}_i))$$

$$P(y_i = j \mid \mathbf{x}_i, \mathbf{z}_i) = \Phi\!\left(\frac{c_j - f(\mathbf{x}_i)}{\sigma_i}\right) - \Phi\!\left(\frac{c_{j-1} - f(\mathbf{x}_i)}{\sigma_i}\right)$$

### Key Result
On imbalanced Amazon Software Review datasets, **HOPANN and OPANN are the only models that predict the minority class**. All other models (ANN, SVM, RF, XGB, Ordered Probit) collapse to the majority class.

In [3]:
# ── Verify all imports work before running component demos ────────────────────
import torch
import numpy as np

try:
    from hopann.models.cutting_points import CuttingPoints
    from hopann.models.mean_network import MeanNetwork
    from hopann.models.variance_network import VarianceNetworkANN, VarianceNetworkLinear
    from hopann.models.opann import OPANN
    from hopann.models.hopann import HOPANN
    from hopann.training.losses import OrderedProbitNLL
    print("All hopann modules imported successfully.")
except ImportError as e:
    print(f"Import failed: {e}")
    print()
    print("Fix: Make sure you ran Cell 2 (Colab) or Cell 2b (local) first.")
    print(f"Current sys.path entries containing 'hopann':")
    for p in sys.path:
        if 'hopann' in p.lower() or 'paper' in p.lower() or 'src' in p.lower():
            print(f"  {p}")

All hopann modules imported successfully.


## Component 1: Cutting Points

Cutting points $c_1 < c_2 < \ldots < c_{J-1}$ must be **strictly ordered** to ensure valid probabilities.

We enforce this via a **cumulative sum + softplus** reparameterisation:
$$c_k = c_1 + \sum_{j=2}^{k} \text{softplus}(\delta_j), \quad k \geq 2$$

This guarantees $c_k > c_{k-1}$ for all $k$ while remaining fully differentiable.

In [5]:
# J=5 classes → J-1=4 cutting points
cp = CuttingPoints(num_classes=5)
cuts = cp()  # forward(): returns ordered [J-1] tensor

print(f"Raw parameters: {cp.raw_params.data.numpy().round(4)}")   # ← fixed
print(f"Ordered cuts:   {cuts.detach().numpy().round(4)}")
print(f"Shape:          {cuts.shape}  (should be [4] for J=5)")
print(f"Strictly increasing: {bool((cuts[1:] > cuts[:-1]).all())}")
print("\nCutting points partition the real line into J=5 regions:")
vals = cuts.detach().numpy()
print(f"  (-inf, {vals[0]:.3f}] → class 1")
for i in range(len(vals)-1):
    print(f"  ({vals[i]:.3f}, {vals[i+1]:.3f}] → class {i+2}")
print(f"  ({vals[-1]:.3f}, +inf) → class {len(vals)+1}")

Raw parameters: [-1.     -0.0537 -0.0537 -0.0537]
Ordered cuts:   [-1.     -0.3333  0.3333  1.    ]
Shape:          torch.Size([4])  (should be [4] for J=5)
Strictly increasing: True

Cutting points partition the real line into J=5 regions:
  (-inf, -1.000] → class 1
  (-1.000, -0.333] → class 2
  (-0.333, 0.333] → class 3
  (0.333, 1.000] → class 4
  (1.000, +inf) → class 5


## Component 2: MeanNetwork

Maps input features $\mathbf{x}_i \in \mathbb{R}^K$ to a **scalar latent index** via a single hidden layer:

$$h_q = \sigma\!\left(\sum_{k=1}^{K} w_{qk} x_{ik} + b_q\right), \quad q = 1, \ldots, Q$$

$$f(\mathbf{x}_i, \theta) = \sum_{q=1}^{Q} v_q h_q + c$$

Output shape: $[B, 1]$.

In [7]:
B, K, Q = 4, 12, 16
net = MeanNetwork(input_dim=K, hidden_dim=Q)   # ← hidden_dim
x = torch.randn(B, K)
out = net(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")
print(f"Output vals:  {out.detach().squeeze().numpy().round(4)}")
n_params = sum(p.numel() for p in net.parameters())
print(f"Parameters:   {n_params}")

Input shape:  torch.Size([4, 12])
Output shape: torch.Size([4, 1])
Output vals:  [-0.7135  0.2068 -0.0628 -0.793 ]
Parameters:   225


## Component 3: VarianceNetwork (HOPANN only)

Maps $\mathbf{z}_i$ (default $\mathbf{z}_i = \mathbf{x}_i$, ASSUMED conf=0.55) to a
**per-sample standard deviation** $\sigma_i > 0$:

$$\sigma_i = \exp(\text{VarianceANN}(\mathbf{z}_i))$$

The $\exp$ output activation guarantees $\sigma_i > 0$ always.

> **Note (ASSUMED, conf=0.62):** The paper describes $\sigma_i = \exp(z_i \gamma)$ — ambiguous between linear and ANN. Both are implemented; switch via config.

In [8]:
B, K = 4, 12
z = torch.randn(B, K)

var_ann = VarianceNetworkANN(input_dim=K, hidden_dim=16)   # ← hidden_dim, no activation arg
sigma_ann = var_ann(z)
print("[ANN variant]")
print(f"  sigma shape: {sigma_ann.shape}")
print(f"  All positive: {bool((sigma_ann > 0).all())}")
print(f"  Range: [{sigma_ann.min().item():.4f}, {sigma_ann.max().item():.4f}]")

var_lin = VarianceNetworkLinear(input_dim=K)
sigma_lin = var_lin(z)
print("\n[Linear variant]")
print(f"  sigma shape: {sigma_lin.shape}")
print(f"  All positive: {bool((sigma_lin > 0).all())}")

[ANN variant]
  sigma shape: torch.Size([4, 1])
  All positive: True
  Range: [0.1798, 0.4362]

[Linear variant]
  sigma shape: torch.Size([4, 1])
  All positive: True


## Component 4: OPANN — Full Model

Combines MeanNetwork + CuttingPoints into a full ordinal classifier:

$$P(y_i = j \mid \mathbf{x}_i) = \Phi(c_j - f(\mathbf{x}_i)) - \Phi(c_{j-1} - f(\mathbf{x}_i))$$

with $c_0 = -\infty$, $c_J = +\infty$.

In [9]:
B, K, J = 4, 12, 5
opann_model = OPANN(input_dim=K, hidden_dim=16, num_classes=J)   # ← hidden_dim
x = torch.randn(B, K)
probs = opann_model(x)
print(f"Input:  {x.shape}")
print(f"Output: {probs.shape}  [B, J]")
print(f"Sums to 1: {probs.sum(dim=1).detach().numpy().round(5)}")

Input:  torch.Size([4, 12])
Output: torch.Size([4, 5])  [B, J]
Sums to 1: [1. 1. 1. 1.]


## Component 5: HOPANN — Full Model

Extends OPANN with per-observation heteroskedasticity:

$$P(y_i = j \mid \mathbf{x}_i, \mathbf{z}_i) = \Phi\!\left(\frac{c_j - f(\mathbf{x}_i)}{\sigma_i}\right) - \Phi\!\left(\frac{c_{j-1} - f(\mathbf{x}_i)}{\sigma_i}\right)$$

When $\sigma_i = 1$ for all $i$, HOPANN reduces exactly to OPANN.

In [13]:
B, K, J = 4, 12, 5
hopann_model = HOPANN(
    input_dim=K,
    hidden_dim=16,
    variance_hidden_dim=16,
    num_classes=J,
    variance_type="ann",
)
x = torch.randn(B, K)
probs = hopann_model(x)  # z is handled internally
print(f"Input shape: {x.shape}")
print(f"Output shape: {probs.shape}")
print(f"Sums to 1: {probs.sum(dim=1).detach().numpy().round(5)}")
n_params = sum(p.numel() for p in hopann_model.parameters())
print(f"Parameters: {n_params:,}")

Input shape: torch.Size([4, 12])
Output shape: torch.Size([4, 5])
Sums to 1: [1. 1. 1. 1.]
Parameters: 454


## Component 6: OrderedProbitNLL — Loss Function

The training objective is the ordered probit negative log-likelihood:

$$\mathcal{L}(\theta) = -\frac{1}{N} \sum_{i=1}^{N} \log P(y_i = y_i^{\text{true}} \mid \mathbf{x}_i)$$

Equivalently: cross-entropy loss applied to the ordered probit probabilities instead of softmax.

In [14]:
import math

B, J = 8, 5
loss_fn = OrderedProbitNLL()

probs   = torch.softmax(torch.randn(B, J), dim=1)  # valid prob vectors
targets = torch.randint(0, J, (B,))                # 0-indexed labels

loss = loss_fn(probs, targets)
print(f"Probs shape:    {probs.shape}")
print(f"Targets:        {targets.tolist()}")
print(f"NLL loss:       {loss.item():.4f}")
print(f"Random baseline NLL ≈ {math.log(J):.4f}  (i.e. log(J) for uniform probs)")
print(f"\nGradient check: loss.requires_grad = {loss.requires_grad}  (must be True)")

Probs shape:    torch.Size([8, 5])
Targets:        [3, 4, 4, 2, 2, 1, 2, 3]
NLL loss:       2.0642
Random baseline NLL ≈ 1.6094  (i.e. log(J) for uniform probs)

Gradient check: loss.requires_grad = False  (must be True)


## Mini-Training Demonstration

Short training loop on **200 synthetic samples** (no downloads). Verifies:
1. Full forward pass works end-to-end
2. Loss decreases over steps
3. Gradients flow through all components

In [15]:
# Synthetic dataset
torch.manual_seed(42)
np.random.seed(42)

N, K, J = 200, 12, 5
X_syn = torch.randn(N, K)
y_syn = torch.randint(0, J, (N,))

n_train = int(0.8 * N)
X_train, X_val = X_syn[:n_train], X_syn[n_train:]
y_train, y_val = y_syn[:n_train], y_syn[n_train:]

print(f"Synthetic dataset: N={N}, K={K}, J={J} classes")
print(f"  Train: {X_train.shape}   Val: {X_val.shape}")
print(f"  Class distribution: {dict(zip(*np.unique(y_syn.numpy(), return_counts=True)))}")

Synthetic dataset: N=200, K=12, J=5 classes
  Train: torch.Size([160, 12])   Val: torch.Size([40, 12])
  Class distribution: {np.int64(0): np.int64(50), np.int64(1): np.int64(45), np.int64(2): np.int64(33), np.int64(3): np.int64(37), np.int64(4): np.int64(35)}


In [16]:
# Mini HOPANN training — 30 steps
hopann_mini = HOPANN(
    input_dim=K,
    hidden_dim=8,
    variance_hidden_dim=8,
    num_classes=J,
    variance_type="ann"
).to(device)

optimizer = torch.optim.Adam(hopann_mini.parameters(), lr=1e-3)
loss_fn   = OrderedProbitNLL()

X_tr = X_train.to(device)
y_tr = y_train.to(device)

n_params = sum(p.numel() for p in hopann_mini.parameters())
print(f"HOPANN mini — {n_params:,} parameters  |  device: {device}\n")
print(f"{'Step':>5}  {'NLL Loss':>10}")
print("-" * 20)

losses = []
hopann_mini.train()
for step in range(1, 31):
    optimizer.zero_grad()
    probs = hopann_mini(X_tr) # z is handled internally
    loss  = loss_fn(probs, y_tr)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if step in (1, 5, 10, 20, 30):
        print(f"{step:>5}  {loss.item():>10.4f}")

trend = "decreasing" if losses[-1] < losses[0] else "NOT decreasing — check model"
print(f"\nLoss trend: {losses[0]:.4f} → {losses[-1]:.4f}  ({trend})")

HOPANN mini — 230 parameters  |  device: cuda

 Step    NLL Loss
--------------------
    1      1.7290
    5      1.7071
   10      1.6842
   20      1.6515
   30      1.6309

Loss trend: 1.7290 → 1.6309  (decreasing)


In [17]:
# Validation + OPANN comparison
opann_mini = OPANN(
    input_dim=K, hidden_dim=8, num_classes=J
).to(device)
opann_opt = torch.optim.Adam(opann_mini.parameters(), lr=1e-3)

opann_mini.train()
for _ in range(30):
    opann_opt.zero_grad()
    loss = loss_fn(opann_mini(X_tr), y_tr)
    loss.backward()
    opann_opt.step()

X_v = X_val.to(device)
y_v = y_val.to(device)

hopann_mini.eval()
opann_mini.eval()
with torch.no_grad():
    h_probs = hopann_mini(X_v)
    o_probs = opann_mini(X_v)
    h_loss  = loss_fn(h_probs, y_v).item()
    o_loss  = loss_fn(o_probs, y_v).item()
    h_acc   = (h_probs.argmax(1) == y_v).float().mean().item()
    o_acc   = (o_probs.argmax(1) == y_v).float().mean().item()

print(f"{'Model':<10} {'Val NLL':>10} {'Val Acc':>10}")
print("-" * 34)
print(f"{'OPANN':<10} {o_loss:>10.4f} {o_acc:>10.4f}")
print(f"{'HOPANN':<10} {h_loss:>10.4f} {h_acc:>10.4f}")
print(f"{'Random':<10} {math.log(J):>10.4f} {1/J:>10.4f}  (baseline)")
print("\n(Synthetic data only — real comparison needs the Amazon dataset)")

Model         Val NLL    Val Acc
----------------------------------
OPANN          1.8138     0.2000
HOPANN         1.5469     0.2500
Random         1.6094     0.2000  (baseline)

(Synthetic data only — real comparison needs the Amazon dataset)


In [18]:
# Paper's reported results (qualitative — from SIR)
results = [
    {"exp": 1, "name": "5-class imbalanced",         "best_f1": "HOPANN/OPANN", "best_acc": "XGB"},
    {"exp": 2, "name": "3-class imbalanced",         "best_f1": "HOPANN/OPANN", "best_acc": "XGB"},
    {"exp": 3, "name": "Modified 3-class imbalanced", "best_f1": "HOPANN/OPANN", "best_acc": "XGB"},
    {"exp": 4, "name": "Modified 3-class balanced",   "best_f1": "OP/ANN",      "best_acc": "RF"},
]

print("Paper Results Summary (Jeong, 2024 — Computational Economics)")
print(f"Dataset: Amazon Software Reviews  |  N=6,173  |  Jan–Sep 2018")
print()
print(f"{'Exp':>4}  {'Name':<32} {'Best F1-macro':<20} {'Best Accuracy'}")
print("-" * 74)
for r in results:
    print(f"{r['exp']:>4}  {r['name']:<32} {r['best_f1']:<20} {r['best_acc']}")

print()
print("KEY FINDING: HOPANN & OPANN are the ONLY models that predict the")
print("minority class in imbalanced datasets. All others collapse to majority.")

Paper Results Summary (Jeong, 2024 — Computational Economics)
Dataset: Amazon Software Reviews  |  N=6,173  |  Jan–Sep 2018

 Exp  Name                             Best F1-macro        Best Accuracy
--------------------------------------------------------------------------
   1  5-class imbalanced               HOPANN/OPANN         XGB
   2  3-class imbalanced               HOPANN/OPANN         XGB
   3  Modified 3-class imbalanced      HOPANN/OPANN         XGB
   4  Modified 3-class balanced        OP/ANN               RF

KEY FINDING: HOPANN & OPANN are the ONLY models that predict the
minority class in imbalanced datasets. All others collapse to majority.


## What to Do Next

### Full reproduction (needs the Amazon dataset)

```bash
# In Colab — after uploading paper-repos.zip
%cd /content/paper-repos

# Generate synthetic data (quick test)
!python data/preprocess.py --synthetic --n-samples 500 --output-path data/amazon_software_reviews.csv

# Train HOPANN, Experiment 1
!python train.py --model hopann --experiment 1

# Full paper reproduction
!python run_experiments.py
```

---

### Implementation Assumptions (from SIR)

| Parameter | Assumed Value | Confidence | Note |
|---|---|---|---|
| `batch_size` | 64 | **Low (0.52)** | Unspecified in paper |
| `learning_rate` | 1e-3 | **Low (0.52)** | Adam default; searched |
| `variance_network_type` | `ann` | **Medium (0.62)** | Ambiguous — switch in `configs/config.yaml` |
| `z_i = x_i` | True | **Medium (0.55)** | Not distinguished from x |

All assumptions are marked `# ASSUMED:` in source files.